# Banks Project - Data Extraction and Transformation

You have been hired as a data engineer by research organization. Your boss has asked you to create a code that can be used to compile the list of the top 10 largest banks in the world ranked by market capitalization in billion USD. Further, the data needs to be transformed and stored in GBP, EUR and INR as well, in accordance with the exchange rate information that has been made available to you as a CSV file. The processed information table is to be saved locally in a CSV format and as a database table.

Your job is to create an automated system to generate this information so that the same can be executed in every financial quarter to prepare the report.

Particulars of the code to be made have been shared below.

## Project Details
Write a function to extract the tabular information from the given URL under the heading By Market Capitalization, and save it to a data frame.

Write a function to transform the data frame by adding columns for Market Capitalization in GBP, EUR, and INR, rounded to 2 decimal places, based on the exchange rate information shared as a CSV file.

Write a function to load the transformed data frame to an output CSV file.

Write a function to load the transformed data frame to an SQL database server as a table.

Write a function to run queries on the database table.

Run the following queries on the database table:

a. Extract the information for the London office, that is Name and MC_GBP_Billion

b. Extract the information for the Berlin office, that is Name and MC_EUR_Billion

c. Extract the information for New Delhi office, that is Name and MC_INR_Billion

Write a function to log the progress of the code.

While executing the data initialization commands and function calls, maintain appropriate log entries.

### Parameters

| **Parameter**                   | **Value**                                                                                                                     |
|----------------------------------|-------------------------------------------------------------------------------------------------------------------------------|
| **Code name**                    | `banks_project.py`                                                                                                           |
| **Data URL**                     | [List of Largest Banks in the World](https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks) |
| **Exchange rate CSV path**       | [Exchange Rate CSV](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-PY0221EN-Coursera/labs/v2/exchange_rate.csv) |
| **Table Attributes (Extraction)**| Name, MC_USD_Billion                                                                                                         |
| **Table Attributes (Final)**     | Name, MC_USD_Billion, MC_GBP_Billion, MC_EUR_Billion, MC_INR_Billion                                                          |
| **Output CSV Path**              | `./Largest_banks_data.csv`                                                                                                    |
| **Database Name**                | `Banks.db`                                                                                                                   |
| **Table Name**                   | `Largest_banks`                                                                                                              |
| **Log File**                     | `code_log.txt`                                                                                                               |



In [1]:
# Import Exchange csv
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-PY0221EN-Coursera/labs/v2/exchange_rate.csv

--2025-03-05 17:01:32--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-PY0221EN-Coursera/labs/v2/exchange_rate.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 198.23.119.245
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|198.23.119.245|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 45 [text/csv]
Saving to: ‘exchange_rate.csv’

exchange_rate.csv   100%[===================>]      45  --.-KB/s    in 0s      

2025-03-05 17:01:33 (20.2 MB/s) - ‘exchange_rate.csv’ saved [45/45]



In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from datetime import datetime
import sqlite3

In [3]:
def log_progress(message):
    timestamp_format = '%Y-%h-%d-%H:%M:%S'
    now = datetime.now()
    timestamp = now.strftime(timestamp_format)
    with open(log_file, "a") as f:
        f.write(timestamp + ',' + message + '\n')

def extract(url, table_attributes):
    page = requests.get(url).text
    data = BeautifulSoup(page,'html.parser')
    df = pd.DataFrame(columns=table_attributes)
    tables = data.find_all('tbody')
    rows = tables[0].find_all('tr')

    for row in rows:
        col = row.find_all('td')
        if len(col) != 0:
            names = col[1].find_all('a')
            name = names[1].contents[0].strip()
            mc_usd_billion = col[2].contents[0].strip()
            new_dict = {'Name': name,
                        'MC_USD_Billion': mc_usd_billion}
            df_tmp = pd.DataFrame(new_dict, index=[0])
            df = pd.concat([df, df_tmp], ignore_index=True)

    return df

def transform(df, csv_path):
    exchanges = pd.read_csv(csv_path)
    df['MC_USD_Billion'] = pd.to_numeric(df['MC_USD_Billion'])
    exchanges.set_index('Currency', inplace=True)
    EUR_exchange = exchanges.loc['EUR','Rate']
    GBP_exchange = exchanges.loc['GBP','Rate']
    INR_exchange = exchanges.loc['INR','Rate']
    df['MC_EUR_Billion'] = round(df.MC_USD_Billion*EUR_exchange,2)
    df['MC_GBP_Billion'] = round(df.MC_USD_Billion*GBP_exchange,2)
    df['MC_INR_Billion'] = round(df.MC_USD_Billion*INR_exchange,2)

    return df

def load_to_csv(df, output_path):
    df.to_csv(output_path)

def load_to_db(df, sql_connection, table_name):
    df.to_sql(table_name, sql_connection, if_exists='replace', index=False)

def run_query(query_statement, sql_connection):
    print(query_statement)
    query_output = pd.read_sql(query_statement, sql_connection)
    print(query_output)

In [4]:
# Features
url = 'https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks'
table_attributes = ['Name','MC_USD_Billion']
csv_path = 'exchange_rate.csv'
output_path = 'largest_bank_data.csv'
log_file = 'code_log.txt'
db_name = 'Banks.db'
table_name = 'Largest_banks'
query_1 = 'SELECT * FROM Largest_banks'
query_2 = 'SELECT AVG(MC_GBP_Billion) FROM Largest_banks'
query_3 = 'SELECT Name from Largest_banks LIMIT 5'

In [5]:
log_progress('Extracting Files')
df = extract(url,table_attributes)

log_progress('Transform Files')
transform(df,csv_path)

log_progress('Save CSV')
load_to_csv(df,output_path)

log_progress('Database Connection')
sql_connection = sqlite3.connect(db_name)

log_progress('Load Content to Databases')
load_to_db(df,sql_connection,table_name)

log_progress('Run Queries')
run_query(query_1,sql_connection)
run_query(query_2,sql_connection)
run_query(query_3,sql_connection)

log_progress('Program Ends')

SELECT * FROM Largest_banks
                                      Name  MC_USD_Billion  MC_EUR_Billion  \
0                           JPMorgan Chase          432.92          402.62   
1                          Bank of America          231.52          215.31   
2  Industrial and Commercial Bank of China          194.56          180.94   
3               Agricultural Bank of China          160.68          149.43   
4                                HDFC Bank          157.91          146.86   
5                              Wells Fargo          155.87          144.96   
6                        HSBC Holdings PLC          148.90          138.48   
7                           Morgan Stanley          140.83          130.97   
8                  China Construction Bank          139.82          130.03   
9                            Bank of China          136.81          127.23   

   MC_GBP_Billion  MC_INR_Billion  
0          346.34        35910.71  
1          185.22        19204.58  
2    